# Other Analysis

Research code accompanying [Şeker (2026), Expert Systems](https://doi.org/10.1111/exsy.70300).
See the [repository README](../README.md) for dependencies, dataset preparation and citation.

This is an exploratory notebook: select the intended dataset and experiment cells before
running. Local intermediate datasets and manual notebook state are required. Stored
outputs and execution counts have been removed. Model and API identifiers retain the
historical experiment settings; check availability before calling external services.


In [ ]:
from pathlib import Path
import os

# Keep relative data/model paths anchored at the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MARC_translate_sentiment_topic_all")
df = dataset.to_pandas()

In [ ]:
df.info()

In [ ]:
df['emotion'].value_counts(normalize=True).plot.pie(autopct='%1.1f%%')

In [ ]:
df.groupby('language')['emotion'].value_counts(normalize=True).unstack().plot.bar(stacked=True, figsize=(12,6))


In [ ]:
import seaborn as sns
sns.boxplot(x='stars', y='sentiment_score', data=df)

In [ ]:
def label_star(star):
    if star in [1, 2]:
        return 'negative'
    elif star == 3:
        return 'neutral'
    else:
        return 'positive'

df['star_sentiment'] = df['stars'].apply(label_star)


In [ ]:
emotion_map = {
    "Joy": "positive",
    "Disappointment": "negative",
    "Dissatisfaction": "negative",
    "Anger": "negative",
    "Mixed": "neutral",
    "Neutral": "neutral"
}

In [ ]:
df["emotion_mapped"] = df["emotion"].map(emotion_map)

In [ ]:
(df["emotion_mapped"] == df["star_sentiment"]).value_counts(normalize=True)

In [ ]:
mismatch = df[df['emotion_mapped'] != df['star_sentiment']]
mismatch['emotion'].value_counts()

In [ ]:
df['is_match'] = df['emotion_mapped'] == df['star_sentiment']
category_accuracy = df.groupby('product_category')['is_match'].mean().sort_values(ascending=False)

In [ ]:
category_accuracy

In [ ]:
lang_accuracy = df.groupby('language')['is_match'].mean().sort_values()
lang_accuracy